In [ ]:
from transformers import pipeline

# load the zero-shot classifier (first run downloads the model — may take a minute)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["opportunity", "risk", "trend"]

text = "Lufthansa faces pilot strikes and rising fuel costs threatening its profits."
result = classifier(text, candidate_labels=labels)

print(result)

{'sequence': 'Lufthansa faces pilot strikes and rising fuel costs threatening its profits.', 'labels': ['a business risk, threat or problem', 'an industry trend or market shift', 'a business opportunity or positive development'], 'scores': [0.8235887289047241, 0.15647804737091064, 0.019933223724365234]}


In [5]:
top_label = result["labels"][0]    # 'risk'  (first = highest score)
top_score = result["scores"][0]    # 0.93    (its confidence)

print("Category:", top_label)
print("Confidence:", round(top_score, 2))

Category: a business risk, threat or problem
Confidence: 0.82


In [6]:
import json
documents = json.load(open("lufthansa_data.json", encoding="utf-8"))

labels = ["opportunity", "risk", "trend"]

# test on the first 5 real docs
for d in documents[:5]:
    result = classifier(d["text"], candidate_labels=labels)
    cat   = result["labels"][0]
    score = result["scores"][0]
    print(f"[{cat}] ({score:.2f})  {d['text'][:90]}")
    print()

[trend] (0.53)  Financial reports - Lufthansa Group Investor Relations. Financial reports 2024 Annual Repo

[trend] (0.46)  Financial reports & publications - Lufthansa Group Investor Relations. The Lufthansa Group

[risk] (0.38)  Financial Data - Lufthansa Technik. Three-quarters of revenue now comes from business with

[trend] (0.41)  Lufthansa - Annual Reports - CompaniesMarketCap.com. Annual Reports Annual Reports Half-ye

[trend] (0.60)  Lufthansa Group Posts Record Revenue, Profit Surge. Mar 6, 2026 · COLOGNE — The Lufthansa 



In [19]:
sentiment_pipe = pipeline("sentiment-analysis",
                          model="cardiffnlp/twitter-roberta-base-sentiment-latest")

#3-class sentiment (negative / neutral / positive)
from collections import Counter

counts = Counter()
for d in documents:
    label = sentiment_pipe(d["text"][:512])[0]["label"]
    d["sentiment"] = label          # store it on the doc (reuse later — no re-running)
    counts[label] += 1

print(counts)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Counter({'neutral': 118, 'positive': 55, 'negative': 27})


In [20]:
from collections import Counter

text_counts = Counter(d["text"] for d in documents)
dupes = {t: c for t, c in text_counts.items() if c > 1}

print("Texts appearing more than once:", len(dupes))
for t, c in list(dupes.items())[:5]:
    print(f"  x{c}: {t[:80]}")

Texts appearing more than once: 3
  x2: Current information - Lufthansa. Get the latest flight information including upd
  x2: Home - Lufthansa Group. The Lufthansa Group is a global aviation group with a to
  x2: Lufthansa | Lufthansa Group. Airport lounges Find out about our airports and gro


In [21]:
labels = ["opportunity", "risk", "trend"]

for d in documents:
    d["category"] = classifier(d["text"], candidate_labels=labels)["labels"][0]

print("Done classifying")

# see the category spread
from collections import Counter
print(Counter(d["category"] for d in documents))

Done classifying
Counter({'trend': 100, 'risk': 87, 'opportunity': 13})


In [22]:
import json
json.dump(documents,
          open("lufthansa_labeled.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saved", len(documents), "labeled docs")

Saved 200 labeled docs
